# Project File Sharing: Break It, See It, Fix It

Files moved into a **My Projects** area from the command line (scp, cp, mv) often become invisible or read-only for other project members. The cause is POSIX: project sharing rides on per-member ACL entries plus an ACL *mask*, and command-line tools either cap the mask with a restrictive file mode or wipe the member entries entirely. Tapis operates on projects as a service account, so it cannot repair files it does not own, which is why this historically required an administrator with sudo.

dapi closes that gap. This notebook demonstrates the full lifecycle on a live project:

1. **create** a file owned by *you* (exactly like an scp'd file),
2. **break** it the way `scp` of a private file does,
3. **audit**: see precisely who can access what and why,
4. **fix** it with one call, `ds.projects.fix_permissions()`,
5. **verify**, then audit the whole project.

The repair works from anywhere (laptop, JupyterHub, CI) because dapi reaches the storage host through the `cloud.data` system, which acts as the calling user, and a file's owner may always repair its ACLs.

In [ ]:
%pip install --quiet --upgrade "dapi @ git+https://github.com/DesignSafe-CI/dapi.git@dev"

**Restart the kernel once after the install** (Kernel → Restart), then run from the next cell.

In [ ]:
import dapi
from dapi import DSClient

assert hasattr(dapi.projects, "fix_project_permissions"), (
    "Old dapi is still loaded. Restart the kernel and rerun from here."
)
ds = DSClient()

PROJECT = "PRJ-6457"  # <-- any project where you are a member

# Resolve the project's path on the storage host (used to act as yourself)
from dapi.projects import resolve_project_uuid

system_id = resolve_project_uuid(ds.tapis, PROJECT)
HOST_ROOT = ds.tapis.systems.getSystem(systemId=system_id).rootDir.strip("/")
print("project system:", system_id)
print("host path     :", "/" + HOST_ROOT)

## 1. Create a file owned by you

Uploading through `cloud.data` writes the file as *your* uid on the storage host, exactly the ownership an scp or cp from a login node produces. (A normal upload through the project system would be owned by the Tapis service account instead.)

In [ ]:
import os
import tempfile

NAME = "issue-demo.txt"

tmp = os.path.join(tempfile.mkdtemp(), NAME)
with open(tmp, "w") as f:
    f.write("permission lifecycle demo\n")
ds.files.upload(tmp, f"tapis://cloud.data/{HOST_ROOT}/{NAME}")
print("created", NAME, "owned by you")

## 2. Break it

`scp` of a mode-600 file caps the ACL **mask** at `---`, which vetoes every member's ACL entry, including the Tapis service account's. Reproduce that exactly:

In [ ]:
ds.tapis.files.setFacl(
    systemId="cloud.data",
    path=f"{HOST_ROOT}/{NAME}",
    operation="ADD",
    recursionMethod="NONE",
    aclString="mask::---",
)
print("mask capped: the file is now invisible to every project member")

## 3. Audit: who can actually see it

`posix_acl` is each member's inherited entry, `mask` is what the file mode did to it, `other` is the world bits, and `effective` is the truth. Expect `none` for everyone, membership looks fine, the mask vetoes it.

In [ ]:
print(ds.projects.permissions(PROJECT, "/" + NAME, output="df").to_string(index=False))

## 4. Fix it with one call

`fix_permissions` tries the strongest repair each file allows: direct ACL repair for service-owned files, the **owner tier** through `cloud.data` for your own files, copy-recreate for files the service account can read, and for another member's file it reports the exact call for that person to run. Every repair is verified against the storage before being reported.

In [ ]:
import json

report = ds.projects.fix_permissions(PROJECT, "/" + NAME)
print(json.dumps(report, indent=1))

## 5. Verify

In [ ]:
print(ds.projects.permissions(PROJECT, "/" + NAME, output="df").to_string(index=False))

## Whole-project audit and repair

The same two calls take a directory (or the project root) and handle everything under it: healthy files are skipped, and the report says what happened to each of the rest. `dry_run=True` previews without changing anything.

In [ ]:
plan = ds.projects.fix_permissions(PROJECT, "/", dry_run=True)
print(json.dumps(plan, indent=1))

In [ ]:
def audit(project):
    for f in ds.projects.files(project, output="raw"):
        print(f"\n=== {f.name}  (mode: {f.nativePermissions}) ===")
        print(
            ds.projects.permissions(project, "/" + f.name, output="df").to_string(
                index=False
            )
        )


audit(PROJECT)

## The complete story

- **Your own broken transfers**: `ds.projects.fix_permissions(PROJECT, "/file")`, from any machine.
- **Tapis-owned files**, including files that predate a newly added member: fixed directly by the same call.
- **Another member's broken transfers**: the report hands you the exact call for them to run.

**Prevention beats repair.** Transfer into projects through Tapis (portal, dapi, or job archiving straight into the project system), or, when using `cp`/`scp` from a login node, finish with `chmod -R g+rwX` on the destination. Never `mv`, `cp -p`, or `rsync -a` into a project, they clone the source's ACLs and wipe the member entries.